# Testes Automatizados com pytest — Tutorial

**COMP — Prof. Dr. André Yoshiaki Kashiwabara** · DCOMP/UFS · `andreyoshiaki@dcomp.ufs.br`

Ao final deste tutorial você será capaz de:

- escrever testes em arquivos `test_*.py` usando `assert` puro;
- rodar a suíte e **ler** o relatório de falha para localizar o defeito;
- testar exceções com `pytest.raises` e floats com `pytest.approx`;
- transformar testes repetidos em uma tabela de casos com `@pytest.mark.parametrize`;
- extrair o preparo repetido para uma *fixture* e compartilhá-la via `conftest.py`.

## Como usar este notebook

Em várias células há a pergunta **"o que vai acontecer?"**. Responda *antes* de executar —
comparar a sua previsão com o resultado real é o que faz o conceito grudar.

Rodamos o pytest a partir do notebook com `!{sys.executable} -m pytest`, que executa
o pytest do mesmo Python do kernel. No seu computador, fora do notebook, o comando é
simplesmente `pytest`.

## 0. Preparando o ambiente

In [ ]:
import os
import pathlib
import subprocess
import sys

# No Colab o pytest ja vem instalado; localmente instalamos se faltar.
try:
    import pytest
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    import pytest

# Todo o material da aula fica em um diretorio proprio.
if pathlib.Path.cwd().name != "aula_pytest":
    destino = pathlib.Path.cwd() / "aula_pytest"
    destino.mkdir(exist_ok=True)
    os.chdir(destino)

print("pytest", pytest.__version__)
print("diretorio de trabalho:", pathlib.Path.cwd())

## 1. O código que vamos testar

Um módulo pequeno de apoio ao lançamento de notas. A célula abaixo **grava** o arquivo
`notas.py` no diretório de trabalho — é o `%%writefile` que faz isso.

In [ ]:
%%writefile notas.py
"""Funcoes de apoio ao lancamento de notas."""


def media(notas):
    """Media aritmetica de uma lista nao vazia de notas."""
    return sum(notas) / len(notas)


def aprovado(nota, minimo=6.0):
    """True se a nota alcanca a media minima de aprovacao."""
    return nota > minimo

## 2. O primeiro teste

Três convenções e nada mais:

1. o arquivo se chama `test_*.py`;
2. cada teste é uma função `test_*`;
3. a verificação é um `assert` comum de Python.

### Preveja antes de rodar

Leia os três testes abaixo junto com o `notas.py` da seção anterior.
**Quantos passam? Qual falha, e por quê?** Anote a sua resposta antes de executar.

In [ ]:
%%writefile test_notas.py
from notas import aprovado, media


def test_media_de_tres_notas():
    assert media([6.0, 7.0, 8.0]) == 7.0


def test_aprovado_acima_da_media():
    assert aprovado(8.0) is True


def test_aprovado_na_nota_exata():
    assert aprovado(6.0) is True

In [ ]:
!{sys.executable} -m pytest -q

## 3. Lendo o relatório de falha

O pytest reescreve o `assert` antes de executá-lo para guardar os valores intermediários.
Por isso o relatório mostra **o que a expressão realmente valia** — repare nas linhas
começadas por `E`.

Duas opções úteis: `-v` dá o nome de cada teste, e `arquivo::teste` roda um teste só.

In [ ]:
!{sys.executable} -m pytest -v

In [ ]:
!{sys.executable} -m pytest "test_notas.py::test_aprovado_na_nota_exata" -q

### Exercício 1 — conserte o código, não o teste

A falha está na fronteira: a nota **exatamente igual** à média mínima deve aprovar.

Edite a célula abaixo (é o `notas.py` inteiro), corrija a função `aprovado` e execute-a
para regravar o arquivo. Depois rode o pytest de novo até a suíte ficar verde.

In [ ]:
%%writefile notas.py
"""Funcoes de apoio ao lancamento de notas."""


def media(notas):
    """Media aritmetica de uma lista nao vazia de notas."""
    return sum(notas) / len(notas)


def aprovado(nota, minimo=6.0):
    """True se a nota alcanca a media minima de aprovacao."""
    # TODO: a nota igual ao minimo tambem aprova. Corrija a comparacao.
    return nota > minimo

In [ ]:
!{sys.executable} -m pytest -q

## 4. A forma de um teste: Arrange–Act–Assert

Todo teste tem três fases: **preparar** os dados, **executar** uma vez o que está sob
teste e **comparar** o resultado com o esperado. Separá-las por linhas em branco torna o
teste legível sem abrir a implementação.

O nome da função é a outra metade da documentação: `test_media_ignora_ordem_das_notas`
diz o comportamento; `test_media_2` não diz nada.

In [ ]:
%%writefile test_aaa.py
from notas import media


def test_media_ignora_ordem_das_notas():
    ordenadas = [5.0, 7.0, 9.0]            # Arrange
    embaralhadas = [9.0, 5.0, 7.0]

    resultado_a = media(ordenadas)          # Act
    resultado_b = media(embaralhadas)

    assert resultado_a == resultado_b       # Assert

In [ ]:
!{sys.executable} -m pytest test_aaa.py -v

## 5. Testando o erro esperado

Lançar exceção faz parte do contrato da função. `pytest.raises` registra isso: o teste
só passa se a exceção indicada acontecer dentro do bloco `with`.

In [ ]:
%%writefile test_erros.py
import pytest

from notas import media


def test_media_de_lista_vazia_falha():
    with pytest.raises(ZeroDivisionError):
        media([])

In [ ]:
!{sys.executable} -m pytest test_erros.py -v

### Exercício 2 — o contrato de erro de `aprovado`

Comparar texto com número levanta `TypeError` em Python: `"dez" >= 6.0` não é uma
comparação válida.

Complete o teste abaixo para documentar que `aprovado("dez")` levanta `TypeError`.

In [ ]:
%%writefile test_erros_exercicio.py
import pytest

from notas import aprovado


def test_aprovado_com_texto_levanta_typeerror():
    # TODO: use "with pytest.raises(...)" para verificar que aprovado("dez") falha
    assert False, "TODO: escreva a verificacao com pytest.raises"

In [ ]:
!{sys.executable} -m pytest test_erros_exercicio.py -v

## 6. Comparando floats: `pytest.approx`

Em ponto flutuante, `0.1 + 0.2` não dá exatamente `0.3`. Sempre que o resultado passar por
divisão ou soma de floats, compare com tolerância.

### Preveja antes de rodar

Dos dois testes abaixo, qual falha?

In [ ]:
%%writefile test_floats.py
import pytest

from notas import media


def test_media_com_igualdade_exata():
    assert media([0.1, 0.2, 0.3]) == 0.2


def test_media_com_approx():
    assert media([0.1, 0.2, 0.3]) == pytest.approx(0.2)

In [ ]:
!{sys.executable} -m pytest test_floats.py -v

## 7. Um teste, muitos casos

Vamos acrescentar ao módulo uma função que converte nota em conceito.
O `-a` do `%%writefile` **acrescenta** ao arquivo em vez de sobrescrever.

In [ ]:
%%writefile -a notas.py


def conceito(nota):
    """Converte a nota em um conceito de A a F."""
    if nota >= 9.0:
        return "A"
    if nota >= 7.5:
        return "B"
    if nota >= 6.0:
        return "C"
    if nota >= 4.0:
        return "D"
    return "F"

Testar cinco faixas com o padrão da seção 4 dá nisto — cinco testes idênticos em estrutura:

In [ ]:
%%writefile test_conceito_repetido.py
from notas import conceito


def test_conceito_A():
    assert conceito(9.5) == "A"


def test_conceito_B():
    assert conceito(8.0) == "B"


def test_conceito_C():
    assert conceito(6.5) == "C"


def test_conceito_D():
    assert conceito(4.0) == "D"


def test_conceito_F():
    assert conceito(2.0) == "F"

In [ ]:
!{sys.executable} -m pytest test_conceito_repetido.py -q

### Exercício 3 — uma tabela de casos

Reescreva os cinco testes acima como **um único teste parametrizado**. O esqueleto está
pronto: preencha a lista de casos e o corpo do teste.

Depois de rodar, repare no relatório: cada linha da tabela continua sendo um teste
independente, identificado entre colchetes.

In [ ]:
%%writefile test_conceito_param.py
import pytest

from notas import conceito


@pytest.mark.parametrize("nota, esperado", [
    (9.5, "A"),
    # TODO: acrescente os casos B, C, D e F
])
def test_conceito_por_faixa(nota, esperado):
    # TODO: verifique que conceito(nota) devolve o esperado
    assert False, "TODO: compare conceito(nota) com esperado"

In [ ]:
!{sys.executable} -m pytest test_conceito_param.py -v

### Exercício 4 — os casos que realmente pegam bugs

As fronteiras são onde os defeitos moram: o valor exato do limite, logo abaixo e logo acima.

Escreva um teste parametrizado para `aprovado` cobrindo `10.0`, `6.0`, `5.99` e `0.0`.
Use `ids` para dar nome a cada caso.

Se rodar antes de preencher a tabela, o pytest reporta `SKIPPED`: sem nenhum caso,
não há nada para executar.

In [ ]:
%%writefile test_fronteiras.py
import pytest

from notas import aprovado


@pytest.mark.parametrize("nota, esperado", [
    # TODO: (nota, True/False) para cada fronteira
], ids=[
    # TODO: um nome legivel por caso
])
def test_aprovacao_nas_fronteiras(nota, esperado):
    # TODO: verifique que aprovado(nota) devolve o esperado
    assert False, "TODO: escreva a verificacao"

In [ ]:
!{sys.executable} -m pytest test_fronteiras.py -v

## 8. Fixtures: o preparo que se repete

Agora um objeto com estado — uma turma com alunos matriculados.

In [ ]:
%%writefile turma.py
"""Uma turma e as notas dos seus alunos."""

from notas import media


class Turma:
    def __init__(self, codigo):
        self.codigo = codigo
        self.alunos = {}

    def matricular(self, nome, notas):
        self.alunos[nome] = list(notas)

    def media_do_aluno(self, nome):
        return media(self.alunos[nome])

    def media_geral(self):
        medias = [self.media_do_aluno(nome) for nome in self.alunos]
        return media(medias)

    def aprovados(self, minimo=6.0):
        from notas import aprovado
        return sorted(nome for nome in self.alunos
                      if aprovado(self.media_do_aluno(nome), minimo))

Testar essa classe do jeito ingênuo repete o *Arrange* inteiro em cada teste:

In [ ]:
%%writefile test_turma_repetido.py
from turma import Turma


def test_media_geral_da_turma():
    turma = Turma("COMP0496")
    turma.matricular("Ana", [8.0, 9.0])
    turma.matricular("Bruno", [6.0, 5.0])
    assert turma.media_geral() == 7.0


def test_aprovados_da_turma():
    turma = Turma("COMP0496")
    turma.matricular("Ana", [8.0, 9.0])
    turma.matricular("Bruno", [6.0, 5.0])
    assert turma.aprovados() == ["Ana"]

In [ ]:
!{sys.executable} -m pytest test_turma_repetido.py -q

### Exercício 5 — extraia a fixture

Uma **fixture** é uma função decorada com `@pytest.fixture` que devolve o cenário pronto.
O teste a recebe declarando um parâmetro **com o mesmo nome da fixture** — é assim que o
pytest sabe o que injetar.

Complete o esqueleto: monte a turma dentro da fixture e escreva os dois testes usando-a.

In [ ]:
%%writefile test_turma_fixture.py
import pytest

from turma import Turma


@pytest.fixture
def turma_com_dois_alunos():
    # TODO: crie a turma, matricule Ana [8.0, 9.0] e Bruno [6.0, 5.0]
    # e devolva o objeto com "return"
    return None


def test_media_geral_da_turma(turma_com_dois_alunos):
    # TODO: verifique que a media geral e 7.0
    assert False, "TODO: complete a fixture e este teste"


def test_aprovados_da_turma(turma_com_dois_alunos):
    # TODO: verifique que apenas Ana esta aprovada
    assert False, "TODO: complete a fixture e este teste"

In [ ]:
!{sys.executable} -m pytest test_turma_fixture.py -v

## 9. Fixtures que já vêm prontas

Não precisa importar nem declarar: basta pedir pelo nome no parâmetro do teste.

- `tmp_path` — um diretório temporário exclusivo daquele teste;
- `capsys` — captura o que o código escreveu na tela;
- `monkeypatch` — troca temporária de atributos e variáveis de ambiente.

O `yield` dentro de uma fixture separa o preparo da limpeza: o que vem depois dele roda
ao final do teste, mesmo se o teste falhar.

In [ ]:
%%writefile test_prontas.py
from turma import Turma


def boletim(nome, media_do_aluno):
    print(f"{nome}: {media_do_aluno:.1f}")


def test_boletim_formata_com_uma_casa(capsys):
    boletim("Ana", 8.456)

    capturado = capsys.readouterr()

    assert capturado.out == "Ana: 8.5\n"


def test_exporta_notas_para_arquivo(tmp_path):
    turma = Turma("COMP0496")
    turma.matricular("Ana", [8.0, 9.0])
    destino = tmp_path / "notas.csv"

    destino.write_text(f"Ana,{turma.media_do_aluno('Ana')}\n")

    assert destino.read_text() == "Ana,8.5\n"

In [ ]:
!{sys.executable} -m pytest test_prontas.py -v

## 10. Compartilhando fixtures: `conftest.py`

Fixtures definidas em `conftest.py` ficam disponíveis para todos os testes daquele
diretório e dos subdiretórios — **sem import**. É o lugar do cenário que mais de um
arquivo de teste precisa.

In [ ]:
%%writefile conftest.py
import pytest

from turma import Turma


@pytest.fixture
def turma_padrao():
    turma = Turma("COMP0496")
    turma.matricular("Ana", [8.0, 9.0])
    turma.matricular("Bruno", [6.0, 5.0])
    return turma

In [ ]:
%%writefile test_usa_conftest.py
# Repare: nenhum import da fixture. O pytest a encontra no conftest.py.


def test_turma_tem_dois_alunos(turma_padrao):
    assert len(turma_padrao.alunos) == 2


def test_media_geral(turma_padrao):
    assert turma_padrao.media_geral() == 7.0

In [ ]:
!{sys.executable} -m pytest test_usa_conftest.py -v

## 11. Marcadores: pular e prever falhas

`skip` pula sempre, `skipif` pula sob condição e `xfail` diz "esperamos que falhe".
Use sempre `reason`: teste pulado sem justificativa é teste esquecido.

No relatório, `s` é pulado e `x` é falha esperada — nenhum dos dois quebra a suíte.

In [ ]:
%%writefile test_marcadores.py
import sys

import pytest


@pytest.mark.skip(reason="exportacao para PDF ainda nao implementada")
def test_exporta_boletim_em_pdf():
    ...


@pytest.mark.skipif(sys.version_info < (3, 10), reason="usa match/case")
def test_classifica_com_match():
    assert True


@pytest.mark.xfail(reason="bug conhecido: arredondamento na terceira casa")
def test_media_com_muitas_casas():
    from notas import media
    assert media([1 / 3, 1 / 3, 1 / 3]) == 0.333

In [ ]:
!{sys.executable} -m pytest test_marcadores.py -v

## 12. Desafio final — a suíte é sua

O módulo `frequencia.py` calcula a frequência do aluno. As duas funções estão **por
implementar**, e não há um único teste ainda.

**Regras:**

- `percentual_de_presenca(presencas, total_de_aulas)` devolve o percentual (0 a 100);
- se `total_de_aulas` for zero, deve levantar `ValueError`;
- `frequencia_suficiente(percentual, minimo=75.0)` diz se o aluno tem frequência
  suficiente — o valor **exatamente igual** ao mínimo já é suficiente.

**A sua tarefa:**

1. escreva primeiro os testes (incluindo as fronteiras e o caso de erro);
2. rode a suíte e veja tudo falhar;
3. implemente as funções até tudo ficar verde;
4. acrescente um teste parametrizado com pelo menos quatro casos.

In [ ]:
%%writefile frequencia.py
"""Calculo de frequencia dos alunos."""


def percentual_de_presenca(presencas, total_de_aulas):
    """Percentual de presenca, de 0 a 100."""
    # TODO: implemente. Lance ValueError se total_de_aulas for zero.
    raise NotImplementedError


def frequencia_suficiente(percentual, minimo=75.0):
    """True se o percentual alcanca o minimo exigido."""
    # TODO: implemente. Cuidado com a fronteira: igual ao minimo e suficiente.
    raise NotImplementedError

In [ ]:
%%writefile test_frequencia.py
import pytest

from frequencia import frequencia_suficiente, percentual_de_presenca


def test_presenca_em_todas_as_aulas():
    # TODO: 20 presencas em 20 aulas devem dar 100.0
    assert False, "TODO: escreva este teste"


# TODO: escreva os demais testes, incluindo:
#   - metade das aulas
#   - total_de_aulas igual a zero (pytest.raises)
#   - a fronteira de 75.0 em frequencia_suficiente
#   - um teste parametrizado com pelo menos quatro casos

In [ ]:
!{sys.executable} -m pytest test_frequencia.py -v

## Checklist da aula

- [ ] Sei escrever `test_*.py` com funções `test_*` e `assert`
- [ ] Sei rodar a suíte e selecionar testes com `-v`, `-k` e `arquivo::teste`
- [ ] Leio o relatório de falha e identifico o valor real antes de mexer no código
- [ ] Separo Arrange, Act e Assert, e dou ao teste um nome que descreve o comportamento
- [ ] Testo exceções com `pytest.raises` e floats com `pytest.approx`
- [ ] Converto testes repetidos em `@pytest.mark.parametrize`, cobrindo as fronteiras
- [ ] Extraio o preparo repetido para uma fixture e compartilho via `conftest.py`

## Referências

- Okken, B. *Python Testing with pytest*. 2. ed. Pragmatic Bookshelf, 2022.
- Aniche, M. *Testes Automatizados de Software: Um Guia Prático*. Casa do Código, 2015.
- Beck, K. *Test-Driven Development: By Example*. Addison-Wesley, 2003.
- Documentação oficial: <https://docs.pytest.org/>

Lista completa em `pytest/referencias.bib`.